# v20.1 Pure Qwen Inference -- Transformers fallback

Patched to skip vLLM installation entirely. Uses Transformers + PEFT + bitsandbytes 4-bit inference to avoid Kaggle vLLM/FlashInfer install/runtime failures.


# Pipeline v20.1 -- PURE QWEN inference/eval
**EXACT 2026 -- data v5 strict-clean**

Final answer path is **pure Qwen3-8B LoRA v20** only:
- no Z3
- no hybrid
- no anti-overclaim
- no LGBM
- BoN self-consistency: N=5, temp=0.5, top_p=0.95


In [1]:
# ==================================================================
# STAGE 0 -- Transformers fallback for Kaggle T4
# PATCH: skip vLLM install completely.
# Reason: pip installing vLLM can hang / conflict on Kaggle images, and vLLM 0.22
# may select FlashInfer and fail JIT. This notebook only needs pure-Qwen inference,
# so Transformers + PEFT + bitsandbytes is the most stable path.
# ==================================================================
import os, sys, subprocess, importlib, shutil, glob

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

print("="*80, flush=True)
print("STAGE 0 -- Transformers fallback, NO vLLM install", flush=True)
print("="*80, flush=True)

def _pip(*args):
    print("pip install", " ".join(args), flush=True)
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"] + list(args),
        capture_output=True, text=True
    )

# Install only lightweight/stable packages needed for PEFT LoRA inference.
# Do not install vLLM here.
for pkg in ["protobuf==5.29.5", "peft", "accelerate", "bitsandbytes", "safetensors"]:
    res = _pip(pkg)
    if res.returncode != 0:
        print("WARNING install issue for", pkg, flush=True)
        print(res.stderr[-1500:], flush=True)

import json, re, csv, time, random, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("\n" + "="*80, flush=True)
print("IMPORT SUMMARY", flush=True)
print("="*80, flush=True)
print(f"PyTorch      : {torch.__version__}", flush=True)
print(f"Transformers : {transformers.__version__}", flush=True)
print(f"CUDA OK      : {torch.cuda.is_available()}", flush=True)
print(f"N_GPUS       : {N_GPUS}", flush=True)
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}        : {props.name} ({props.total_memory/1024**3:.1f} GB)", flush=True)
print("Mode         : Transformers + PEFT 4-bit fallback", flush=True)
print("All imports OK", flush=True)


STAGE 0 vLLM compatibility installer
Pre-installed vLLM: FAIL (No module named 'vllm')
  Trying transformers==4.48.0 + vllm... WARNING 06-06 19:46:29 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
OK

IMPORT SUMMARY
PyTorch      : 2.11.0+cu130
CUDA OK      : True
N_GPUS       : 2
GPU 0        : Tesla T4 (14.6 GB)
GPU 1        : Tesla T4 (14.6 GB)
vLLM version : 0.22.1
Transformers : 4.57.6
All imports OK


In [2]:

# ==================================================================
# BOOT CHECK
# ==================================================================
import os, sys, glob, torch
print("="*70)
print("BOOT CHECK")
print("="*70)
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "CUDA available=", torch.cuda.is_available())
print("N_GPUS:", N_GPUS)
print("/kaggle/input entries:")
for p in sorted(glob.glob("/kaggle/input/*"))[:30]:
    print(" -", p)
print("="*70)


BOOT CHECK
Python: 3.12.13
Torch : 2.11.0+cu130 CUDA available= True
N_GPUS: 2
/kaggle/input entries:
 - /kaggle/input/datasets
 - /kaggle/input/models
 - /kaggle/input/notebooks


In [3]:
# ==================================================================
# LOCATE + UNZIP LoRA adapter
# Updated Kaggle notebook-output paths:
#   v17: https://www.kaggle.com/code/yixuanisthebest/v17-finetune-lora
#   v20: https://www.kaggle.com/code/nguyenminhtric/v20-finetune
# Default: v20. Change LORA_VERSION = "v17" only for ablation/comparison.
# ==================================================================
import os, glob, zipfile, shutil

LORA_VERSION = "v20"   # "v20" for final candidate, "v17" for previous aggressive baseline
UNZIP_DEST = "/kaggle/working"

LORA_CANDIDATES = {
    "v20": {
        "zip_names": ["qwen3_cot_lora_v20_v5.zip", "qwen3_cot_lora_v20*.zip"],
        "adapter_dir_hint": "qwen3_cot_lora_v20_v5",
        "exact_paths": [
            # Kaggle Notebook Output path from: https://www.kaggle.com/code/nguyenminhtric/v20-finetune
            "/kaggle/input/notebooks/nguyenminhtric/v20-finetune/qwen3_cot_lora_v20_v5.zip",
            # Sometimes Kaggle mounts notebook outputs without /notebooks/<user>/ prefix
            "/kaggle/input/v20-finetune/qwen3_cot_lora_v20_v5.zip",
            "/kaggle/input/nguyenminhtric/v20-finetune/qwen3_cot_lora_v20_v5.zip",
        ],
    },
    "v17": {
        "zip_names": ["qwen3_cot_lora_v17.zip", "qwen3_cot_lora_v17*.zip"],
        "adapter_dir_hint": "qwen3_cot_lora_v17",
        "exact_paths": [
            # Kaggle Notebook Output path from: https://www.kaggle.com/code/yixuanisthebest/v17-finetune-lora
            "/kaggle/input/notebooks/yixuanisthebest/v17-finetune-lora/qwen3_cot_lora_v17.zip",
            # Sometimes Kaggle mounts notebook outputs without /notebooks/<user>/ prefix
            "/kaggle/input/v17-finetune-lora/qwen3_cot_lora_v17.zip",
            "/kaggle/input/yixuanisthebest/v17-finetune-lora/qwen3_cot_lora_v17.zip",
        ],
    },
}

assert LORA_VERSION in LORA_CANDIDATES, f"Unknown LORA_VERSION={LORA_VERSION}"
spec = LORA_CANDIDATES[LORA_VERSION]

print("="*70)
print(f"LOCATE LoRA {LORA_VERSION}")
print("="*70)

# Build robust candidate list: exact notebook-output paths first, then recursive globs.
zip_candidates = []
zip_candidates.extend(spec["exact_paths"])
for name in spec["zip_names"]:
    zip_candidates.extend([
        f"/kaggle/input/**/{name}",
        f"/kaggle/input/notebooks/**/{name}",
    ])

def _find_existing_zip(cands):
    hits = []
    for p in cands:
        if any(ch in p for ch in "*?["):
            hits.extend(sorted(glob.glob(p, recursive=True)))
        elif os.path.exists(p):
            hits.append(p)
    # prefer exact notebook path; then shorter paths; stable unique order
    seen, uniq = set(), []
    for h in hits:
        if h not in seen:
            seen.add(h); uniq.append(h)
    return uniq

zip_hits = _find_existing_zip(zip_candidates)
print("zip hits:")
for h in zip_hits[:10]:
    print(" -", h)

resolved = None
if zip_hits:
    zpath = zip_hits[0]
    print("Found zip:", zpath)
    # remove old extracted adapter for this version, then unzip cleanly
    hinted = os.path.join(UNZIP_DEST, spec["adapter_dir_hint"])
    if os.path.exists(hinted):
        shutil.rmtree(hinted, ignore_errors=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(UNZIP_DEST)

    # expected folder first
    if os.path.exists(os.path.join(hinted, "adapter_config.json")):
        resolved = hinted
    else:
        # fallback: scan working dir for an adapter folder matching version hint
        for root, _, files in os.walk(UNZIP_DEST):
            if "adapter_config.json" in files and spec["adapter_dir_hint"].replace("_v5", "") in root:
                resolved = root
                break
else:
    print("No zip found; searching for unzipped adapter folders in /kaggle/input...")
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files and spec["adapter_dir_hint"].replace("_v5", "") in root:
            resolved = root
            break

assert resolved is not None, (
    f"Cannot find LoRA adapter for {LORA_VERSION}. "
    "Make sure the Kaggle inference notebook has added the corresponding notebook output as Data."
)
assert os.path.exists(os.path.join(resolved, "adapter_config.json")), resolved
assert os.path.exists(os.path.join(resolved, "adapter_model.safetensors")), resolved

COT_LORA_PATH = resolved
print("[OK] LORA_VERSION =", LORA_VERSION)
print("[OK] COT_LORA_PATH =", COT_LORA_PATH)
print("[OK] adapter_model.safetensors MB =", os.path.getsize(os.path.join(resolved, "adapter_model.safetensors"))/1e6)

LOCATE LoRA v20
zip hits: ['/kaggle/input/notebooks/nguyenminhtric/v20-finetune/qwen3_cot_lora_v20_v5.zip']
Found zip: /kaggle/input/notebooks/nguyenminhtric/v20-finetune/qwen3_cot_lora_v20_v5.zip
[OK] COT_LORA_PATH = /kaggle/working/qwen3_cot_lora_v20_v5
[OK] adapter_model.safetensors MB = 174.655536


In [4]:

# ==================================================================
# CONFIG + PATH RESOLVER
# ==================================================================
import os, glob, random, json, re, csv, time
from collections import Counter, defaultdict

def _resolve(cands, label="path"):
    expanded = []
    for p in cands:
        hits = sorted(glob.glob(p, recursive=True)) if any(ch in p for ch in "*?[") else [p]
        expanded.extend(hits)
    for p in expanded:
        if os.path.exists(p):
            print(f"resolved {label}: {p}")
            return p
    print(f"WARNING {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

DATASET_PATH = _resolve([
    "/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "DATASET")

TEST_PATH = _resolve([
    "/kaggle/input/**/generated_v4style_300.json",
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")

QWEN_MODEL_ID = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"

SEED = 42
TRAIN_RATIO = 0.80
QWEN_BON_N = 5
QWEN_BON_TEMP = 0.5
QWEN_TOP_P = 0.95
COT_MAX_TOKENS = 768
TENSOR_PARALLEL = min(max(N_GPUS, 1), 2)
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTIL = 0.85
OUTPUT_JSON = "/kaggle/working/pipeline_results_20_1_pureqwen_v5.json"
SUBMISSION_CSV = "/kaggle/working/submission_v20_pureqwen.csv"
SUBMISSION_JSON = "/kaggle/working/submission_v20_pureqwen.json"

V20_COT_SYSTEM = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)

print("="*70)
print("PURE QWEN CONFIG")
print("="*70)
print("DATASET_PATH:", DATASET_PATH)
print("TEST_PATH   :", TEST_PATH)
print("COT_LORA_PATH:", COT_LORA_PATH)
print("QWEN_BON_N:", QWEN_BON_N, "temp:", QWEN_BON_TEMP, "top_p:", QWEN_TOP_P)
print("TENSOR_PARALLEL:", TENSOR_PARALLEL)
print("NO Z3 / NO HYBRID / NO LGBM")


resolved DATASET: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
resolved TEST: /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
PURE QWEN CONFIG
DATASET_PATH: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
TEST_PATH   : /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
COT_LORA_PATH: /kaggle/working/qwen3_cot_lora_v20_v5
QWEN_BON_N: 5 temp: 0.5 top_p: 0.95
TENSOR_PARALLEL: 2
NO Z3 / NO HYBRID / NO LGBM


In [5]:
# ==================================================================
# STAGE 0B -- Confirm FlashInfer is disabled
# Do NOT force FLASHINFER here. This notebook uses Triton attention on T4.
# ==================================================================
import os, shutil, glob

os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
shutil.rmtree("/root/.cache/torch_extensions", ignore_errors=True)

print("FlashInfer remains disabled; using TRITON_ATTN for Kaggle T4.")
print("If vLLM was imported before this cell in the same kernel, restart and Run All.")


FlashInfer cache cleared; env set.


In [6]:
# ==================================================================
# LOAD TRANSFORMERS MODEL + TOKENIZER + PEFT LoRA
# ==================================================================
import os, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("="*70)
print("Loading Transformers 4-bit model + LoRA...")
print("="*70)

print("torch cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, COT_LORA_PATH)
model.eval()

# Keep aliases so later cells can be version-agnostic.
llm = None
LORA_REQ = None

print(f"Transformers model loaded in {(time.time()-t0):.1f}s")
print("LoRA enabled:", COT_LORA_PATH)
print("Main device:", getattr(model, "device", "device_map_auto"))


Loading vLLM engine...
torch cuda: True
device_count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
INFO 06-06 19:46:37 [utils.py:278] non-default args: {'tokenizer': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1', 'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'model': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'}
WARNING 06-06 19:46:37 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND
WARNING 06-06 19:46:37 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 06-06 19:46:53 [model.py:617] Resolved architecture: Qwen3ForCausalLM
WARNING 06-06 19:46:53 [model.py:2090] Casting torch.bfloat16 to torch.float16.
INFO 06-06 19:46:53 [model.py:1752] Using max model len 8192
INFO 06-06 19:46:53 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(Worker_TP1 pid=264) INFO 06-06 19:47:26 [weight_utils.py:884] Prefetching checkpoint files into page cache started (in background, num_threads=8, block_size=16777216 bytes)
(Worker_TP0 pid=263) INFO 06-06 19:47:34 [weight_utils.py:856] Prefetching checkpoint files: 10% (1/3)


Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:17<01:11, 17.81s/it]


(Worker_TP0 pid=263) INFO 06-06 19:47:44 [weight_utils.py:856] Prefetching checkpoint files: 20% (2/3)
(Worker_TP1 pid=264) INFO 06-06 19:47:45 [weight_utils.py:856] Prefetching checkpoint files: 10% (1/2)


...

(Worker_TP1 pid=264) INFO 06-06 19:47:48 [weight_utils.py:856] Prefetching checkpoint files: 20% (2/2)
(Worker_TP1 pid=264) INFO 06-06 19:47:48 [weight_utils.py:879] Prefetching checkpoint files into page cache finished in 22.24s
(Worker_TP0 pid=263) INFO 06-06 19:47:49 [weight_utils.py:856] Prefetching checkpoint files: 30% (3/3)
(Worker_TP0 pid=263) INFO 06-06 19:47:49 [weight_utils.py:879] Prefetching checkpoint files into page cache finished in 22.95s


Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:25<00:13,  6.83s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:27<00:05,  5.07s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:28<00:00,  3.65s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:28<00:00,  5.78s/it]
(Worker_TP0 pid=263) 


(Worker_TP0 pid=263) INFO 06-06 19:47:55 [default_loader.py:397] Loading weights took 28.97 seconds
(Worker_TP0 pid=263) INFO 06-06 19:47:55 [punica_selector.py:20] Using PunicaWrapperGPU.
(Worker_TP0 pid=263) INFO 06-06 19:47:56 [model_runner.py:295] Model loading took 7.71 GiB and 30.665026 seconds
(Worker_TP1 pid=264) INFO 06-06 19:47:56 [model_runner.py:295] Model loading took 7.71 GiB and 30.687143 seconds
(Worker_TP0 pid=263) WARNING 06-06 19:48:02 [utils.py:279] Using default LoRA kernel configs
(Worker_TP0 pid=263) INFO 06-06 19:48:16 [gpu_worker.py:466] Available KV cache memory: 3.92 GiB
(EngineCore pid=237) INFO 06-06 19:48:16 [kv_cache_utils.py:1733] GPU KV cache size: 57,120 tokens
(EngineCore pid=237) INFO 06-06 19:48:16 [kv_cache_utils.py:1734] Maximum concurrency for 8,192 tokens per request: 6.97x
(Worker_TP0 pid=263) INFO 06-06 19:48:17 [kernel_warmup.py:97] Warming up FlashInfer attention.
(Worker_TP1 pid=264) INFO 06-06 19:48:17 [kernel_warmup.py:97] Warming up Flas

(EngineCore pid=237) Process EngineCore:
(EngineCore pid=237) Traceback (most recent call last):
(EngineCore pid=237)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=237)     self.run()
(EngineCore pid=237)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=237)     self._target(*self._args, **self._kwargs)
(EngineCore pid=237)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1169, in run_engine_core
(EngineCore pid=237)     raise e
(EngineCore pid=237)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1139, in run_engine_core
(EngineCore pid=237)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=237)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=237)   File "/usr/local/lib/python3.12/dist-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=23

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:

# ==================================================================
# DATA LOADING + PROMPT BUILDING
# ==================================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

raw = load_json(DATASET_PATH)
print(f"Loaded labeled data: {len(raw)} records")

rng = random.Random(SEED)
ids = list(range(len(raw)))
rng.shuffle(ids)
n_train = int(len(raw) * TRAIN_RATIO)
train_ids = set(ids[:n_train])
val_ids = set(ids[n_train:])

def iter_questions(samples, split_name, keep_gold=True):
    rows = []
    for sid, s in enumerate(samples):
        nls = s.get("premises-NL", [])
        qs = s.get("questions", [])
        ans = s.get("answers", [])
        exps = s.get("explanation", [])
        for q_idx, q in enumerate(qs):
            gold = str(ans[q_idx]).strip() if keep_gold and q_idx < len(ans) else None
            exp  = str(exps[q_idx]) if q_idx < len(exps) else ""
            rows.append({
                "sample_id": sid, "q_idx": q_idx, "split": split_name,
                "premises_nl": nls, "question": q, "gold": gold, "explanation": exp
            })
    return rows

train_rows = iter_questions([raw[i] for i in range(len(raw)) if i in train_ids], "train", True)
val_rows   = iter_questions([raw[i] for i in range(len(raw)) if i in val_ids], "val", True)
all_rows   = iter_questions(raw, "all", True)

print(f"Train questions: {len(train_rows)}")
print(f"Val questions  : {len(val_rows)}")
print(f"All questions  : {len(all_rows)}")
print("Val label distribution:", Counter(r["gold"] for r in val_rows))

def build_user_msg(premises_nl, question):
    lines = ["### Premises"]
    for i, p in enumerate(premises_nl):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"

def to_chat_prompt(row):
    messages = [
        {"role": "system", "content": V20_COT_SYSTEM},
        {"role": "user", "content": build_user_msg(row["premises_nl"], row["question"])},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:

# ==================================================================
# PURE QWEN BoN GENERATION + ANSWER EXTRACTION
# Transformers fallback version, no vLLM.
# ==================================================================
VALID = {"YES":"Yes", "NO":"No", "UNKNOWN":"Unknown", "A":"A", "B":"B", "C":"C", "D":"D"}

def normalize_answer(x):
    if x is None:
        return "Unknown"
    s = str(x).strip()
    u = s.upper()
    if u in VALID:
        return VALID[u]
    if u in ("TRUE", "T"):
        return "Yes"
    if u in ("FALSE", "F"):
        return "No"
    if "UNKNOWN" in u or "CANNOT BE DETERMINED" in u or "NOT ENOUGH" in u:
        return "Unknown"
    return "Unknown"

def extract_final_answer(text):
    if not text:
        return "Unknown"
    pats = [
        r"Final\s*Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b",
        r"Final\s*answer\s*is\s*(Yes|No|Unknown|A|B|C|D)\b",
        r"Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b",
    ]
    for pat in pats:
        m = re.search(pat, text, flags=re.I)
        if m:
            return normalize_answer(m.group(1))
    tail = text[-300:]
    hits = re.findall(r"\b(Yes|No|Unknown|A|B|C|D)\b", tail, flags=re.I)
    return normalize_answer(hits[-1]) if hits else "Unknown"

def majority_vote(ans_list):
    norm = [normalize_answer(a) for a in ans_list]
    cnt = Counter(norm)
    best_n = max(cnt.values()) if cnt else 0
    tied = {k for k,v in cnt.items() if v == best_n}
    for a in norm:
        if a in tied:
            return a, cnt
    return "Unknown", cnt

@torch.inference_mode()
def generate_candidates_transformers(prompts):
    """Generate QWEN_BON_N candidates for each prompt using Transformers.
    Returns list[list[str]] with shape len(prompts) x QWEN_BON_N.
    """
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_MODEL_LEN,
    )
    # For device_map='auto', put inputs on the first CUDA device.
    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}

    gen = model.generate(
        **enc,
        max_new_tokens=COT_MAX_TOKENS,
        do_sample=True,
        temperature=QWEN_BON_TEMP,
        top_p=QWEN_TOP_P,
        num_return_sequences=QWEN_BON_N,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    prompt_len = enc["input_ids"].shape[1]
    gen_new = gen[:, prompt_len:]
    flat_texts = tokenizer.batch_decode(gen_new, skip_special_tokens=True)

    grouped = []
    for i in range(0, len(flat_texts), QWEN_BON_N):
        grouped.append(flat_texts[i:i+QWEN_BON_N])
    return grouped

def batch_predict(rows, batch_size=4):
    # batch_size is deliberately small for T4 16GB with 4-bit 8B + num_return_sequences=5.
    prompts = [to_chat_prompt(r) for r in rows]
    results = []
    t0 = time.time()

    for start in range(0, len(prompts), batch_size):
        end = min(start + batch_size, len(prompts))
        try:
            grouped_texts = generate_candidates_transformers(prompts[start:end])
        except torch.cuda.OutOfMemoryError:
            print("CUDA OOM at batch_size", batch_size, "→ retrying one-by-one", flush=True)
            torch.cuda.empty_cache()
            grouped_texts = []
            for p in prompts[start:end]:
                grouped_texts.extend(generate_candidates_transformers([p]))

        for row, texts in zip(rows[start:end], grouped_texts):
            ans_list = [extract_final_answer(t) for t in texts]
            pred, votes = majority_vote(ans_list)
            rec = dict(row)
            rec.update({
                "qwen_answer": pred,
                "qwen_votes": dict(votes),
                "qwen_sc_conf": max(votes.values()) / max(1, sum(votes.values())),
                "qwen_raw_0": texts[0] if texts else "",
            })
            results.append(rec)

        print(f"[{end}/{len(prompts)}] elapsed={(time.time()-t0)/60:.1f} min", flush=True)

    return results


In [ ]:

# ==================================================================
# RUN EVAL ON TRAIN / VAL / FULL
# ==================================================================
train_pred = batch_predict(train_rows)
val_pred   = batch_predict(val_rows)
# Avoid re-running full separately: combine train+val predictions.
all_pred = train_pred + val_pred

def _U(x): return str(x).strip().upper()

def compute_metrics(rows):
    preds = [_U(r["qwen_answer"]) for r in rows]
    golds = [_U(r["gold"]) for r in rows]
    labels = sorted(set(golds))
    per = {}
    for lab in labels:
        tp = sum(1 for p,g in zip(preds,golds) if p==lab and g==lab)
        fp = sum(1 for p,g in zip(preds,golds) if p==lab and g!=lab)
        fn = sum(1 for p,g in zip(preds,golds) if p!=lab and g==lab)
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        per[lab] = {"precision":prec, "recall":rec, "f1":f1, "n":sum(1 for g in golds if g==lab)}
    acc = sum(1 for p,g in zip(preds,golds) if p==g) / len(rows) if rows else 0.0
    macro_f1 = sum(v["f1"] for v in per.values()) / len(per) if per else 0.0
    weighted_f1 = sum(v["f1"]*v["n"] for v in per.values()) / len(rows) if rows else 0.0
    mc_labels = {"A","B","C","D"}
    mc_rows = [r for r in rows if _U(r["gold"]) in mc_labels]
    ynu_rows = [r for r in rows if _U(r["gold"]) in {"YES","NO","UNKNOWN"}]
    return {
        "acc": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1, "per_class": per,
        "mc_acc": (sum(_U(r["qwen_answer"])==_U(r["gold"]) for r in mc_rows)/len(mc_rows) if mc_rows else 0.0),
        "ynu_acc": (sum(_U(r["qwen_answer"])==_U(r["gold"]) for r in ynu_rows)/len(ynu_rows) if ynu_rows else 0.0),
        "n": len(rows),
    }

m_train = compute_metrics(train_pred)
m_val   = compute_metrics(val_pred)
m_full  = compute_metrics(all_pred)

def f1(m, lab):
    return m["per_class"].get(lab, {}).get("f1", 0.0)

print("\n" + "="*100)
print("v20.1 PURE QWEN -- metrics")
print("="*100)
print(f"{'Split':8s} | {'acc':>7s} {'mF1':>7s} {'wF1':>7s} {'Yes-F1':>7s} {'No-F1':>7s} {'Unk-F1':>7s} {'MC-acc':>7s}")
print("-"*100)
for name,m in [("TRAIN",m_train),("VAL",m_val),("FULL",m_full)]:
    print(f"{name:8s} | {m['acc']*100:6.1f}% {m['macro_f1']:7.3f} {m['weighted_f1']:7.3f} "
          f"{f1(m,'YES'):7.3f} {f1(m,'NO'):7.3f} {f1(m,'UNKNOWN'):7.3f} {m['mc_acc']*100:6.1f}%")
print("="*100)

print("\nPer-class VAL:")
for lab, d in m_val["per_class"].items():
    print(f"  {lab:>8s}: f1={d['f1']:.3f}  prec={d['precision']:.3f}  rec={d['recall']:.3f}  n={d['n']}")


In [ ]:

# ==================================================================
# SAVE RESULTS
# ==================================================================
payload = {
    "meta": {
        "version": "v20_1_pure_qwen_v5",
        "model": "Qwen3-8B + LoRA v20",
        "answer_path": "pure_qwen_only",
        "no_z3_no_hybrid_no_lgbm": True,
        "seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "qwen_bon_n": QWEN_BON_N,
        "temperature": QWEN_BON_TEMP,
        "top_p": QWEN_TOP_P,
        "lora_version": LORA_VERSION,
        "cot_lora_path": COT_LORA_PATH,
        "dataset_path": DATASET_PATH,
    },
    "metrics": {"train": m_train, "val": m_val, "full": m_full},
    "val_predictions": val_pred,
    "train_predictions_sample": train_pred[:20],
}
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_JSON)

# Decision gate
val = m_val
yes_f1 = f1(val, "YES")
no_f1 = f1(val, "NO")
unk_f1 = f1(val, "UNKNOWN")
macro = val["macro_f1"]
print("\n" + "="*70)
print("V20 DECISION GATE")
print("="*70)
print(f"macro-F1   = {macro:.3f}  target >= 0.620")
print(f"No-F1      = {no_f1:.3f}  target >= 0.600")
print(f"Unknown-F1 = {unk_f1:.3f}  target >= 0.550")
print(f"Yes-F1     = {yes_f1:.3f}  target >= 0.750")
if macro >= 0.62 and no_f1 >= 0.60 and unk_f1 >= 0.55 and yes_f1 >= 0.75:
    print("PASS: v20 is a strong final candidate.")
elif macro < 0.538:
    print("FAIL: below v18 macro-F1. Fallback: v20b No 4x, Unknown 1x.")
elif yes_f1 < 0.70:
    print("WARNING: Yes-F1 low. Fallback: reduce No oversample to 2x.")
elif unk_f1 < 0.50:
    print("WARNING: Unknown-F1 low. Fallback: No 3x, Unknown 1.5x.")
else:
    print("MIXED: compare against v17 aggressive and paper defensibility.")
print("="*70)


In [ ]:

# ==================================================================
# OPTIONAL: PREDICT OFFICIAL-LIKE TEST IF PRESENT
# ==================================================================
if os.path.exists(TEST_PATH):
    try:
        test_raw = load_json(TEST_PATH)
        test_rows = iter_questions(test_raw, "test", keep_gold=False)
        print(f"Loaded test: {len(test_raw)} records / {len(test_rows)} questions")
        test_pred = batch_predict(test_rows)
        # CSV: id,answer. Use sample_id_qidx as stable id unless official format differs.
        with open(SUBMISSION_CSV, "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f)
            w.writerow(["id", "answer"])
            for r in test_pred:
                w.writerow([f"{r['sample_id']}_{r['q_idx']}", r["qwen_answer"]])
        with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
            json.dump(test_pred, f, ensure_ascii=False, indent=2)
        print("Saved:", SUBMISSION_CSV)
        print("Saved:", SUBMISSION_JSON)
    except Exception as e:
        print("Test prediction skipped due to error:", repr(e))
else:
    print("TEST_PATH not found, skip official-like prediction.")
